# verses

> Read printed verse numbers, put verses into chapters, mend their numbering, and repair malformed Devanagari

In [ ]:
#| default_exp verses

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import re
from copy import deepcopy
from difflib import SequenceMatcher
from fastcore.all import AttrDict, L

A vision model reads each page of a book and returns a list of blocks. A block is one piece of the page as the model saw it: a verse, a chapter heading, a commentary, a colophon (the line that closes a chapter, इति ... अध्यायः), or a stretch of prose. Each block has a `kind` that says which of these it is, the printed number `n` if it has one, and its `text`. A page is a dict `{page, blocks}`, which is what `vishalakshi.read_doc` returns.

Readers do not all use the same names. vedicreader's saved readings keep a page's blocks under `items` instead of `blocks`, and some models call a heading `head`. `blocks` returns a page's blocks under either key, and `kind` returns a block's kind with `head` read as `heading`, through the `KINDS` table.

The functions in this module put verses into chapters. A verse comes out as `{n, text, page, heading}`: its number, its text, the page it was printed on, and the heading printed just before it, if any. Whenever a function changes a number or a text, it adds a line to `probs`, a list you pass in, so that a proofreader can check each change against the scan.

In [ ]:
#| export
KINDS = dict(head='heading')
def kind(b) -> str|None:
    "A block's kind, with an alias from `KINDS`, such as `head`, read as the kind it stands for."
    k = b.get('kind'); return KINDS.get(k, k)

def _bk(p): return 'items' if 'items' in p and 'blocks' not in p else 'blocks'
def blocks(p) -> list:
    "A page's list of blocks, whether the page keeps them under `blocks` or under `items`."
    if p.get(k := _bk(p)) is None: p[k] = []
    return p[k]

In [ ]:
test_eq(blocks(dict(page=1, items=[1])), [1]); test_eq(blocks(dict(page=1, blocks=[2], items=[1])), [2]); test_eq(blocks(dict(page=1)), [])
test_eq(kind(dict(kind='head')), 'heading'); test_eq(kind(dict(kind='verse')), 'verse'); test_eq(kind({}), None)

## Verse numbers

A verse number can be printed in ASCII, Devanagari or Tamil digits, and is often wrapped in daṇḍas or brackets, as in ॥ २६ ॥ or (௪௦). `vnum` finds the digits and returns the number as an int, or None when there are no digits. Older Tamil prints write numbers with the signs ௰ (ten), ௱ (hundred) and ௲ (thousand), and `vnum` reads those too: ௬௱௨ is 602.

Sometimes a block carries two numbers, as in २६॥२७. This usually means the model read two verses as one block, so `vnum` returns the last number, 27. If you pass `probs`, it also records that verses 26 and 27 were printed as one. Some books print a running number followed by the verse's number within its section, as in २६॥ १. For those, `pick='first'` returns the running number. The tests below show each case.

In [ ]:
#| export
_DIG = str.maketrans('०१२३४५६७८९௦௧௨௩௪௫௬௭௮௯', '01234567890123456789')
_SIGNS = {'௰': 10, '௱': 100, '௲': 1000}
def _tamil(s):
    "A run of digits and the old Tamil signs ௰ ten, ௱ hundred, ௲ thousand as an int: ௬௱௨ is 602, ௨௱௩ 203."
    tot, d = 0, None
    for c in s:
        if c.isdecimal(): d = (d or 0) * 10 + int(c)
        else: tot += (d or 1) * _SIGNS[c]; d = None
    return tot + (d or 0)

def vnum(n, page=None, probs:list=None, pick:str='last') -> int|None:
    "A printed verse number as an int, or None when it has no digits; when several are printed together, the last one, or the first with `pick='first'`."
    xs = [_tamil(x) for x in re.findall(r'[\d௰௱௲]+', str('' if n is None else n).translate(_DIG))]
    if len(xs) > 1 and probs is not None: probs.append(f'{page}: verses {"-".join(map(str, xs))} printed as one, numbered {xs[-1 if pick == "last" else 0]}')
    return (xs[-1] if pick == 'last' else xs[0]) if xs else None

In [ ]:
_p = []
test_eq(vnum('॥ २६ ॥'), 26); test_eq(vnum('(௪௦)'), 40); test_eq(vnum(12), 12); test_eq(vnum(None), None)
test_eq(vnum('२६॥२७', 'p5', _p), 27); test_eq(_p, ['p5: verses 26-27 printed as one, numbered 27'])
test_eq(vnum('௬௱௨'), 602); test_eq(vnum('௨௱௩'), 203); test_eq(vnum('௰'), 10)
test_eq(vnum('२६॥ १', pick='first'), 26)   # a running number, then the count within its part

## Mending a run

Within a chapter the numbers should run 1, 2, 3 without gaps. Two common mistakes break that run in a way the neighbouring numbers can repair, and `mend` repairs both.

The first is a number read twice. If the model reads 148, 150, 150, 151, the verse after 148 has to be 149, so `mend` takes the first 150 as 149. The second is two verses read as one block. If the model reads verse 148 and then a long block numbered 150, verse 149 is missing and is probably the first half of that block. `mend` splits the block after its last inner line that ends with a daṇḍa (।, ॥ or |) or a full stop, keeping at least three lines in the first part, and numbers that part 149. Both repairs happen only when the verse before is numbered exactly two less.

That rule suits verses printed on four lines. In a book of two-line verses, both lines of a verse end in a daṇḍa, so the last such line is the wrong place to cut. `pair=True` lets the first part be two lines long and cuts where the two parts come out closest in length.

`renumber` repairs the mistakes found in long chapters. One is a run read at a constant offset: in 52, 63, 64, 65, 56 the model misread the tens digit of three verses, and the sequence returns to normal at 56. Another is a single number that disagrees with both neighbours, such as 9 between 2 and 4, or a number that lost its leading digit, such as 1 read for 101 between 100 and 102. `renumber` only changes numbered verses. A verse with no number stays where it is.

In [ ]:
#| export
_END = re.compile(r'[.।॥|]\s*$')
def _ln(vs): return next((v['n'] for v in reversed(vs) if v['n'] is not None), None)
def mend(vs, probs:list, pair:bool=False) -> list:
    "Correct a verse number read twice and split two verses read as one block, where the numbers around them show what is right."
    out = []
    for i, v in enumerate(vs):
        if v['n'] is None: out.append(v); continue
        n, prev = v['n'], _ln(out) or 0
        nxt = vs[i + 1]['n'] if i + 1 < len(vs) else None
        if n == nxt and prev == n - 2:
            out.append(dict(v, n=n - 1)); probs.append(f"{v['page']}: first of two {n}s taken as {n - 1}"); continue
        ls = v['text'].split('\n')
        cut = [j for j, l in enumerate(ls[:-1]) if _END.search(l) and j >= (1 if pair else 2)]
        if prev == n - 2 and cut:
            j = (min(cut, key=lambda c: abs(len(''.join(ls[:c + 1])) - len(''.join(ls[c + 1:])))) if pair else cut[-1]) + 1
            out += [dict(v, n=n - 1, text='\n'.join(ls[:j])), dict(v, text='\n'.join(ls[j:]), heading=None)]
            probs.append(f"{v['page']}: {n} split after its line {j}, the first part taken as {n - 1}"); continue
        out.append(v)
    return out

def renumber(vs, where, probs:list) -> list:
    "Correct misread verse numbers in a chapter from the numbers around them, leaving unnumbered verses where they are."
    ix, out = [i for i, v in enumerate(vs) if v['n'] is not None], [dict(v) for v in vs]
    for i, v in zip(ix, _renum([vs[i] for i in ix], where, probs)): out[i] = v
    return out

def _renum(vs, where, probs):
    vs = [dict(v) for v in vs]
    for i in range(1, len(vs)):
        want = vs[i - 1]['n'] + 1; d = vs[i]['n'] - want
        if not d: continue
        j = i
        while j < len(vs) and j - i < 15 and vs[j]['n'] - d == want + (j - i): j += 1
        if i < j < len(vs) and vs[j]['n'] == want + (j - i):
            for k in range(i, j): probs.append(f"{where}: verse read as {vs[k]['n']} numbered {vs[k]['n'] - d} ({vs[k]['page']})"); vs[k]['n'] -= d
    out = []
    for i, v in enumerate(vs):
        n = v['n']; want = out[-1]['n'] + 1 if out else n
        nxt = vs[i + 1]['n'] if i + 1 < len(vs) else None
        k = 10 ** len(str(n))
        if n != want and (nxt == want + 1 or (n < want and want % k == n % k)):
            probs.append(f"{where}: verse read as {n} numbered {want} ({v['page']})"); n = want
        out.append(dict(v, n=n))
    return out

In [ ]:
V = lambda n, t='a ।\nb ॥', pg=1: dict(n=n, text=t, page=pg, heading=None)
_p = []
test_eq([v['n'] for v in mend([V(148), V(150), V(150), V(151)], _p)], [148, 149, 150, 151]); test_eq(len(_p), 1)
_p = []
_m = mend([V(1), V(3, 'a\nb\nc ॥\nd\ne ॥')], _p)
test_eq([(v['n'], v['text']) for v in _m], [(1, 'a ।\nb ॥'), (2, 'a\nb\nc ॥'), (3, 'd\ne ॥')])
_p = []
test_eq([v['n'] for v in renumber([V(n) for n in (52, 63, 64, 65, 56, 57)], 'ch 3', _p)], [52, 53, 54, 55, 56, 57]); test_eq(len(_p), 3)
test_eq([v['n'] for v in renumber([V(n) for n in (99, 100, 1, 102)], 'ch 9', [])], [99, 100, 101, 102])
test_eq([v['n'] for v in renumber([V(None), V(99), V(100), V(1), V(102)], 'ch 9', [])], [None, 99, 100, 101, 102])   # an unnumbered opener stays put
test_eq([v['n'] for v in mend([V(None), V(1), V(3, 'a\nb\nc ॥\nd\ne ॥')], [])], [None, 1, 2, 3])

## Chapters

`assemble` reads the pages in order and builds chapters, each a dict `{heading, colophon, verses}`. It returns the chapters and a list of problems.

The `kinds` argument says which blocks count as verse. By default these are blocks of kind `verse` and blocks with no kind. With `kinds=None`, every block except headings and colophons counts as verse, which suits a reading in which the model gave verse text kind names of its own.

A heading becomes the name of the open chapter if that chapter has no name yet, or opens a new chapter if none is open. Each verse also keeps the heading printed just before it in its own `heading` field. A verse numbered 1 after a higher number starts a new chapter. A colophon ends the open chapter and is stored as its `colophon`.

A verse that starts at the foot of one page often has its number printed only at its end, on the next page, so the model returns the first part as a block with no number. `assemble` holds every unnumbered block and adds its text to the next numbered verse. If a colophon comes while text is still held, that text becomes the chapter's last verse, numbered one more than the verse before it, and this is recorded in the problems.

A book photographed as two-page spreads may show the same page twice, and then the model reads the same verse twice. A numbered verse whose number and last line match a verse already in the chapter is dropped, and an unnumbered block that repeats the one held before it is kept once. As verses are added, any break in the numbering, such as verse 5 after verse 3, is listed in the problems. When every page is in, `mend` runs on each chapter.

Many books print a commentary after each verse; in Tamil books this is the urai. The model gives such a block a kind of its own, such as `urai`, `gloss` or `bhashya`. Name those kinds in `notes`, and `assemble` keeps each such block with the verse it follows, in `verse['notes'][kind]`, as a list of texts. A commentary that arrives while verse text is still held, with no number yet, is kept for the next verse that is added. With the default `kinds` and no `notes`, these blocks are skipped. Some books, the Vidhānamālai among them, print the number at the end of the commentary and leave the verse unnumbered. With `number_from_note=True`, a numbered commentary block gives its number to the verse text held before it, and this is recorded in the problems.

A page break can also cut a block in two, so that its second half opens the next page. Some readings mark that second half with `cont`. Only when you pass `cont=True` does `assemble` first join each such half to the block it continues, using `join_cont` on a copy of the pages.

Not every unnumbered block belongs to the next verse. Many books open with a maṅgala, a verse of blessing printed before verse 1 without a number, and commentaries quote whole verses from other works. Such a block ends in ॥ like a finished verse. By default `assemble` treats it like any other unnumbered block and adds its text to the next numbered verse, which then holds two verses. With `openers=True`, an unnumbered block that ends in ॥ is kept as a verse of its own, with `n` set to None. It goes into the chapter of the numbered verse that follows it, just before that verse; if a colophon comes first, it goes at the end of the chapter the colophon closes. `mend` and `renumber` leave such verses where they are.

`join_cont` is the joining step on its own. When the first block on a page is marked `cont` and has the same kind as the last block on the page before, its text is added to that block: after a line break for a verse, and after a space for anything else. If the earlier text ends in a hyphen, the hyphen is removed and the two parts are joined with no space, so a word split across the page break is whole again. If the second half carries a number, the joined block takes it. `join_cont` changes the pages it is given and returns them.

The tests below build small pages for each of these cases: a chapter closed by a colophon, a verse read twice, commentary kept as notes, a reading with `items` and `cont`, and maṅgala verses kept with `openers`.

In [ ]:
#| export
def join_cont(pages) -> list:
    "Join each block marked `cont` at the top of a page to the block it continues on the page before, rejoining a word hyphenated at the break."
    last = None
    for p in pages:
        bl = blocks(p)
        if bl and last is not None and bl[0].get('cont') and kind(bl[0]) == kind(last):
            b = bl.pop(0); a, t = (last.get('text') or '').rstrip(), (b.get('text') or '').strip()
            last['text'] = a[:-1] + t if a.endswith('-') else a + ('\n' if kind(b) == 'verse' else ' ') + t
            if b.get('n') is not None: last['n'] = b['n']
        if bl: last = bl[-1]
    return pages

In [ ]:
_jc = [dict(page=1, items=[dict(kind='prose', text='a long com-'), dict(kind='verse', text='क ।')]),
       dict(page=2, items=[dict(kind='verse', text='ख ॥२॥', n=2, cont=True), dict(kind='prose', text='next')]),
       dict(page=3, items=[dict(kind='prose', text='one', cont=True)])]
test_eq([(b['kind'], b['text'], b.get('n')) for p in join_cont(_jc) for b in p['items']], [('prose', 'a long com-', None), ('verse', 'क ।\nख ॥२॥', 2), ('prose', 'next one', None)])
test_eq(join_cont([dict(page=1, blocks=[dict(kind='prose', text='com-')]), dict(page=2, blocks=[dict(kind='prose', text='mentary', cont=True)])])[0]['blocks'][0]['text'], 'commentary')

In [ ]:
#| export
def _alike(a, b, at=.8): return SequenceMatcher(None, a, b).ratio() >= at

def _take(n, t, pg, buf, have, probs):
    "The verse `{n, text, page}` a block closes; None when it has no number and joins `buf`, False when it repeats one of `have`."
    if (n := vnum(n, pg, probs) if n is not None else None) is None:
        if not (buf and _alike(buf[-1], t)): buf.append(t)
        return None
    if any(v['n'] == n and _alike(v['text'].split('\n')[-1], t.split('\n')[-1], .6) for v in have): return False
    v = dict(n=n, text='\n'.join(buf + [t]), page=pg); buf.clear()
    return v

def assemble(pages, pair:bool=False, kinds=('verse', None), notes=(), number_from_note:bool=False, cont:bool=False, openers:bool=False) -> tuple:
    """Build chapters `[{heading, colophon, verses}]` from pages of blocks, and return them with the problems found. `kinds` names the
    block kinds that count as verse, and None means every kind except headings and colophons. A block whose kind is in `notes` is kept in
    the `notes` of its verse, and `number_from_note=True` numbers a verse from its commentary. `cont=True` first joins blocks cut by a page
    break, and `openers=True` keeps an unnumbered block ending in ॥ as a verse of its own with `n` None."""
    if cont: pages = join_cont(deepcopy(list(pages)))
    chs, cur, buf, probs, hd, pend, opn = [], None, [], [], None, {}, []
    new = lambda h=None: dict(heading=h, colophon=None, verses=[])
    def add(n, text, pg):
        nonlocal cur, hd, pend
        if cur is None or (cur['verses'] and n == 1 and cur['verses'][-1]['n'] > 1):
            if cur and cur['verses']: chs.append(cur)
            cur = new()
        if cur['verses'] and n != cur['verses'][-1]['n'] + 1: probs.append(f"{pg}: verse {n} after {cur['verses'][-1]['n']}")
        v = dict(n=n, text=text, page=pg, heading=hd)
        if pend: v['notes'], pend = pend, {}
        cur['verses'] += opn + [v]; opn.clear(); hd = None
        return v
    for p in pages:
        for b in p.get(_bk(p)) or []:
            k, t, n, pg = kind(b), b.get('text') or '', b.get('n'), p.get('page')   # text as read: a leading break or an empty block is the reader's
            if k in notes:
                m = vnum(n, pg, probs) if n is not None else None
                if number_from_note and buf and m is not None:
                    add(m, '\n'.join(buf), pg); buf = []; probs.append(f'{pg}: verse numbered {m} from its {k}')
                last = None if buf else opn[-1] if opn else cur['verses'][-1] if cur and cur['verses'] else None
                (last.setdefault('notes', {}) if last else pend).setdefault(k, []).append(t)
                continue
            if k == 'start':   # a chapter opens here: `repairs.cut_by_plan`
                if cur and cur['verses']: chs.append(cur)
                cur = None; continue
            if k == 'heading':
                hd = t
                if cur is None: cur = new(t)
                elif not cur['heading']: cur['heading'] = t
                continue
            if k == 'colophon':
                if buf and cur and cur['verses']:
                    n = cur['verses'][-1]['n'] + 1
                    cur['verses'].append(dict(n=n, text='\n'.join(buf), page=pg, heading=None)); buf = []
                    probs.append(f'{pg}: unnumbered verse before the colophon taken as {n}')
                if opn: cur = cur or new(); cur['verses'] += opn; opn.clear()
                (cur or new())['colophon'] = t
                if cur: chs.append(cur); cur = None
                continue
            if kinds is not None and k not in kinds: continue
            if openers and vnum(n) is None and t.rstrip().endswith('॥'):
                opn.append(dict(n=None, text='\n'.join(buf + [t]), page=pg, heading=hd)); buf.clear(); hd = None; continue
            if (v := _take(n, t, pg, buf, cur['verses'] if cur else [], probs)) is False: hd = None
            elif v: add(v['n'], v['text'], pg)
    if opn: cur = cur or new(); cur['verses'] += opn
    if cur and cur['verses']: chs.append(cur)
    if buf: probs.append(f'unfinished verse at the end: {buf[-1][:40]}')
    for c in chs: c['verses'] = mend(c['verses'], probs, pair)
    return chs, probs

def check_counts(chs, counts) -> list:
    "List the chapters whose number of verses differs from the expected count, and a difference in the number of chapters."
    out = [] if len(chs) == len(counts) else [f'{len(chs)} chapters, the plan has {len(counts)}']
    return out + [f"chapter {i + 1}: {len(c['verses'])} verses, the plan has {k}" for i, (c, k) in enumerate(zip(chs, counts)) if len(c['verses']) != k]

In [ ]:
B = lambda kind, text, n=None: dict(kind=kind, text=text, n=n)
_pages = [dict(page=0, blocks=[B('heading', 'अथ प्रथमोऽध्यायः'), B('verse', 'क ।\nख ॥१॥', '१'), B('verse', 'ग ।', None)]),
          dict(page=1, blocks=[B('verse', 'घ ॥२॥', '॥२॥'), B('verse', 'ङ ॥३॥', 3), B('colophon', 'इति प्रथमोऽध्यायः')]),
          dict(page=2, blocks=[B('verse', 'च ॥१॥', 1), B('verse', 'छ ॥२॥', 2), B('verse', 'छ ॥२॥', 2), B('prose', 'an English note')])]
_chs, _pr = assemble(_pages)
test_eq(len(_chs), 2); test_eq(_chs[0]['heading'], 'अथ प्रथमोऽध्यायः'); test_eq(_chs[0]['colophon'], 'इति प्रथमोऽध्यायः')
test_eq([(v['n'], v['text']) for v in _chs[0]['verses']], [(1, 'क ।\nख ॥१॥'), (2, 'ग ।\nघ ॥२॥'), (3, 'ङ ॥३॥')])
test_eq([v['n'] for v in _chs[1]['verses']], [1, 2]); test_eq(_pr, [])
test_eq(check_counts(_chs, [3, 2]), []); test_eq(check_counts(_chs, [3, 3]), ['chapter 2: 2 verses, the plan has 3'])

In [ ]:
_u = [dict(page=0, blocks=[B('verse', 'பா', 1), B('urai', 'உரை', 1)])]
test_eq(len(assemble(_u)[0][0]['verses']), 1); test_eq([v['text'] for v in assemble(_u, kinds=None)[0][0]['verses']], ['பா', 'உரை'])   # every kind but headings and colophons

In [ ]:
_vm = [dict(page=0, blocks=[B('verse', 'பா ஒன்று', None), B('urai', 'உரை ஒன்று', 1), B('verse', 'பா இரண்டு', None), B('urai', 'உரை இரண்டு', 2)])]
_c, _p = assemble(_vm, notes=('urai',), number_from_note=True)
test_eq([(v['n'], v['text'], v['notes']['urai']) for v in _c[0]['verses']], [(1, 'பா ஒன்று', ['உரை ஒன்று']), (2, 'பா இரண்டு', ['உரை இரண்டு'])])
_g = [dict(page=0, blocks=[B('verse', 'a ॥१॥', 1), B('gloss', 'its gloss'), B('verse', 'b ॥२॥', 2)])]
test_eq(assemble(_g, notes=('gloss',))[0][0]['verses'][0]['notes'], {'gloss': ['its gloss']})
test_eq('notes' in assemble(_g)[0][0]['verses'][0], False)   # without `notes` a gloss is not a verse and is not kept

In [ ]:
_it = [dict(page=0, items=[B('head', 'अथ प्रथमोऽध्यायः'), B('verse', 'क ।\nख ॥१॥', 1), dict(kind='verse', text='ग ।', open=True)]),
       dict(page=1, items=[dict(kind='verse', text='घ ॥२॥', n=2, cont=True)])]
_c, _p = assemble(_it, cont=True)
test_eq((_c[0]['heading'], [(v['n'], v['text']) for v in _c[0]['verses']]), ('अथ प्रथमोऽध्यायः', [(1, 'क ।\nख ॥१॥'), (2, 'ग ।\nघ ॥२॥')]))
test_eq(_it[1]['items'][0]['text'], 'घ ॥२॥')   # the pages are left as they were

In [ ]:
_op = [dict(page=0, blocks=[B('heading', 'अथ प्रथमोऽध्यायः'), B('verse', 'वन्दे गुरुम् ॥'), B('verse', 'क ।\nख ॥१॥', 1), B('verse', 'ग ॥२॥', 2)]),
       dict(page=1, blocks=[B('verse', 'नमः शिवाय ॥'), B('verse', 'धर्मक्षेत्रे ॥१॥', 1), B('verse', 'समवेता ।'), B('verse', 'मामकाः ॥२॥', 2), B('verse', 'श्रीः ॥'), B('colophon', 'इति')])]
_c, _p = assemble(_op, openers=True)
test_eq([[(v['n'], v['text'], v['heading']) for v in c['verses']] for c in _c],
        [[(None, 'वन्दे गुरुम् ॥', 'अथ प्रथमोऽध्यायः'), (1, 'क ।\nख ॥१॥', None), (2, 'ग ॥२॥', None)],
         [(None, 'नमः शिवाय ॥', None), (1, 'धर्मक्षेत्रे ॥१॥', None), (2, 'समवेता ।\nमामकाः ॥२॥', None), (None, 'श्रीः ॥', None)]])
test_eq(_p, [])
test_eq([(v['n'], v['text'][:5]) for v in assemble(_op)[0][0]['verses']], [(1, 'वन्दे'), (2, 'ग ॥२॥')])   # by default an opener joins the verse after it

## Chapters cut at their colophons

Some editions print no usable chapter headings and number each chapter only in its colophon. The Khemraj edition of the Bṛhat Parāśara Horā Śāstra is one. It has a pūrva khaṇḍa (first part) of 51 adhyāyas (chapters) and an uttara bhāga (second part) of 20, and a chapter's number appears only in its closing line, such as इति ... राशिभेदनामाध्यायः ॥ ३ ॥. `assemble_colophon` cuts the pages into chapters at these colophons. Each chapter comes out as `{part, no, name, heading, colophon, verses}`. By default every block other than a heading or a colophon counts as verse (`kinds=None`).

A colophon closes a chapter only if it contains `adh`, which is ध्याय by default, as in अध्यायः. The chapter's number is the first number after the last `adh` in the colophon. Failing that, it is the colophon block's own `n`, and failing that, one more than the previous chapter in the same part. The chapter's name is the text after one of the `anchor` words, such as जातके or होरायां, and before नाम, or before the ordinal word and अध्याय. If the colophon gives no name, the chapter takes one from an earlier heading of the form अथ ... अध्यायः.

`parts` lists the book's parts in order, each with a pattern. The pages start in the first part, and a heading or colophon that matches a later part's pattern moves the count into that part, so chapter numbers can start again at 1 in the uttara bhāga.

These editions also print things that are not the text, and there is a pattern for each. A colophon matching `contents` closes a table of contents printed before the text, so everything collected up to it is thrown away. A heading matching `skip` starts the commentator's own verses, such as his maṅgala or his praise of the author, and verses are skipped until a heading matching `resume`. Commentators also write colophons of their own. A colophon matching `tika` (टीका or व्याख्या) that repeats the number of the chapter just closed marks the verses between the two colophons as commentary, and they are dropped. An ordinary colophon that repeats the number adds those verses to that chapter instead.

If the verses before a colophon start again at 1 partway through, the chapter before them lost its colophon. The function then splits the verses where the numbering restarts, giving the first part the number before and no colophon or name, unless a chapter with that number already exists. Unnumbered blocks are held and joined to the next numbered verse, and repeated verses are dropped, as in `assemble`. With `pair=True`, text still held when a colophon arrives becomes the chapter's last verse, and `mend` uses its rule for two-line verses. Finally each chapter goes through `renumber` and `mend`, and any break left in its numbering is listed in the problems.

The test below runs a small two-page book through these cases.

In [ ]:
#| export
_UTT = r'त्तरभाग|त्तरार्ध|त्तरार्द्ध|उ० भाग|उ० भा'   # the uttara bhāga, never विंशोत्तरी
_SKIP = r'टीकाकर्तु|टीकाकार|ग्रंथकर्तृप्रशंसा|ग्रन्थकर्तृप्रशंसा'
_ANCHOR = r'(?:सारांशे|सारांश|सा०|भागे|उत्तरार्धे|उत्तरार्द्धे|शास्त्रे|होरायाम्|होरायां|चिन्तामणौ|जातके)'

def _cnum(t, adh):
    "The chapter number a colophon prints after its last `adh` (॥ १५॥), or None."
    ms = list(re.finditer(adh, t))
    return vnum(t[ms[-1].start():], pick='first') if ms else None

def _cname(t, anchor, adh):
    "A chapter's name from its colophon: what stands after `anchor` and before नाम, else before the ordinal and `adh`."
    t = re.sub(r'-\s*\n\s*|-\s+', '', t).replace('\n', ' ')
    m = re.search(anchor + r'\s*(.+?)\s*(?:नाम|ना०)', t) or re.search(anchor + r'\s*(.+?)\s*(?:\S*ो\s*)?ऽ?(?:' + adh + ')', t)
    x = re.sub(r'\s+', ' ', m[1]).strip(' ।-') if m else ''
    return re.sub(r'ा$', '', x) or None

def assemble_colophon(pages, pair:bool=False, kinds=None, parts=(('पूर्व', None), ('उत्तर', _UTT)), adh:str=r'ध्याय', tika:str=r'टीका|व्याख्या',
                      skip:str=_SKIP, resume:str=r'ग्रंथारंभ|ग्रन्थारम्भ|ध्याय', contents:str=r'नुक्रमणिका', anchor:str=_ANCHOR) -> tuple:
    "Cut pages into chapters `[{part, no, name, heading, colophon, verses}]` at the colophons that number them, and return them with the problems found."
    chs, cur, buf, probs, hd, kh, skipping, ath = [], [], [], [], None, parts[0][0], False, None
    part = lambda t, kh: next((nm for nm, r in parts[1:] if re.search(r, t)), kh)
    def close(no, t, pg):
        nonlocal cur
        if chs and (chs[-1]['part'], chs[-1]['no']) == (kh, no):
            if tika and re.search(tika, t): probs.append(f"{pg}: {len(cur)} verses between the colophon of {kh} {no} and its commentator's colophon dropped as the commentary's")
            else: chs[-1]['verses'] += cur; probs.append(f'{pg}: colophon of {kh} {no} repeated; the verses before it kept in that chapter')
        else:
            r = next((i for i in range(1, len(cur)) if cur[i]['n'] == 1 and cur[i - 1]['n'] > 1), None)
            if r is not None and not any((c['part'], c['no']) == (kh, no - 1) for c in chs):
                chs.append(dict(part=kh, no=no - 1, name=None, heading=None, colophon=None, verses=cur[:r]))
                probs.append(f'{pg}: {kh} {no - 1} has no colophon; split from {no} where its numbering restarts'); cur = cur[r:]
            nm = _cname(t, anchor, adh) or ath
            chs.append(dict(part=kh, no=no, name=nm, heading=nm, colophon=t, verses=cur))
        cur = []
    for p in pages:
        pg = p.get('page')
        for b in p.get(_bk(p)) or []:
            k, t, n = kind(b), b.get('text') or '', b.get('n')
            if k == 'heading':
                kh = part(t, kh)
                if skip and re.search(skip, t): skipping = True
                elif re.search(resume, t): skipping = False
                if t.startswith('अथ') and re.search(adh, t): ath = re.sub(r'^अथ\s*|(?:' + adh + ').*$', '', t).strip(' ा।')
                hd = t; continue
            if k == 'colophon':
                if contents and re.search(contents, t): cur, buf, kh = [], [], parts[0][0]; continue
                if not re.search(adh, t): continue
                kh = part(t, kh)
                no = _cnum(t, adh) or vnum(n) or ((chs[-1]['no'] + 1) if chs and chs[-1]['part'] == kh else 1)
                if pair and buf and cur:
                    cur.append(dict(n=cur[-1]['n'] + 1, text='\n'.join(buf), page=pg, heading=None)); probs.append(f"{pg}: unnumbered verse before the colophon taken as {cur[-1]['n']}")
                if cur: close(no, t, pg)
                buf, skipping = [], False; continue
            if skipping or (kinds is not None and k not in kinds): continue
            if v := _take(n, t, pg, buf, cur, probs): cur.append(dict(v, heading=hd)); hd = None
    if cur: probs.append(f'{len(cur)} verses after the last colophon, not placed')
    for c in chs:
        w = f"{c['part']} {c['no']}"
        c['verses'] = vs = mend(renumber(c['verses'], w, probs), probs, pair)
        probs += [f"{w}: verse {b['n']} after {a['n']} ({b['page']})" for a, b in zip(vs, vs[1:]) if b['n'] != a['n'] + 1]
    return chs, probs

In [ ]:
_bj = [dict(page=1, items=[B('colophon', 'इति ग्रन्थानुक्रमणिका'), B('head', 'टीकाकर्तुर्मङ्गलम्'), B('verse', 'टीका ॥१॥', 1), B('head', 'अथ ग्रन्थारम्भः'),
                           B('verse', 'क ॥१॥', 1), B('verse', 'ख ॥२॥', 2), B('colophon', 'इति होरायां संज्ञाध्यायः प्रथमः ॥१॥')]),
       dict(page=2, items=[B('verse', 'रामो राजमणिः ॥१॥', 1), B('verse', 'सदा विजयते ॥२॥', 2), B('verse', 'गणेशं नमामि ॥१॥', 1), B('colophon', 'इति शास्त्रे राशिभेदनामाध्यायः॥ ३॥'),
                           B('head', 'अथोत्तरभागः'), B('verse', 'च ॥१॥', '१'), B('colophon', 'इति उत्तरभागे फलनाम प्रथमोऽध्यायः ॥१॥'),
                           B('colophon', 'इति टीकायां प्रथमोऽध्यायः ॥१॥')])]
_c, _p = assemble_colophon(_bj)
test_eq([(c['part'], c['no'], c['name'], [v['n'] for v in c['verses']]) for c in _c],
        [('पूर्व', 1, 'संज्ञ', [1, 2]), ('पूर्व', 2, None, [1, 2]), ('पूर्व', 3, 'राशिभेद', [1]), ('उत्तर', 1, 'फल', [1])])
assert any('no colophon; split' in x for x in _p)
test_eq(_cname('इति श्रीवराहमिहिरकृते बृहज्जातके वियोनिजन्माध्यायस्तृतीयः॥३॥', _ANCHOR, 'ध्याय'), 'वियोनिजन्म'); test_eq(_cnum('निषेकाध्यायश्चतुर्थः॥ ४॥', 'ध्याय'), 4)

## Akṣaras

An akṣara is one written syllable: a consonant or a cluster of consonants with its vowel sign, or a vowel letter on its own. Some sequences of Unicode characters can never make a correct akṣara, and they turn up when a model writes Devanagari or when a text was typed or converted from an old font. One example is a virāma (्, the mark that removes a consonant's vowel) followed by a vowel sign, as in ध्ा. Another is two vowel signs on one consonant.

`AKSHARA_BAD` names five such faults, each with a pattern, and `bad_aksharas` lists the names of those found in a text. `fix_aksharas` repairs the ones whose right form is not in doubt:

- an i-sign (ि) typed after a virāma is moved to the end of its consonant cluster, so श्िच becomes श्चि;
- a virāma before a vowel sign or an anusvāra is removed, and so is a visarga after a virāma;
- ट्ः followed by a vowel letter becomes ठ with that vowel's sign, so कण्ट्ःए becomes कण्ठे;
- a nukta after a vowel sign is removed;
- े or ै followed by ा becomes ो or ौ;
- a vowel letter followed by a sign that makes another vowel, such as अ with ा, becomes that vowel's letter, here आ;
- a vowel sign typed twice, and a few pairs such as ु followed by ू, become one sign.

The Taittirīya texts write ग्ं for a nasal, so a virāma before an anusvāra after ग is left as it is.

In [ ]:
#| export
_C = '[\u0915-\u0939\u0958-\u095f]'   # consonants, the nukta letters too
_SIGN = {'अ': '', 'आ': 'ा', 'इ': 'ि', 'ई': 'ी', 'उ': 'ु', 'ऊ': 'ू', 'ए': 'े', 'ऐ': 'ै', 'ओ': 'ो', 'औ': 'ौ'}
AKSHARA_BAD = {'virāma before a vowel sign': re.compile('्[ा-ौॢॣ]'),
               'two vowel signs': re.compile('[\u093e-\u0944\u0946-\u094c][\u093e-\u0944\u0946-\u094c]'),
               'virāma before anusvāra or visarga': re.compile('(?<!ग)्[ंः]'),
               'nukta after a vowel sign': re.compile('[ा-्]़'),
               'vowel sign after a vowel letter': re.compile('[ऄ-औ][ा-ौ]')}

def bad_aksharas(t:str) -> list:
    "The names, from `AKSHARA_BAD`, of the malformed akṣaras found in `t`."
    return [k for k, r in AKSHARA_BAD.items() if r.search(t or '')]

def fix_aksharas(t:str) -> str:
    "Return the Devanagari text `t` with the malformed akṣaras whose right form is certain repaired."
    if not t: return t
    t = re.sub(f'्ि((?:{_C}्)*{_C})', lambda m: '्' + m[1] + 'ि', t)
    t = re.sub('ट्ः([अआइईउऊएऐओऔ])', lambda m: 'ठ' + _SIGN[m[1]], t)
    t = re.sub('्([ा-ौॢॣ])', r'\1', t)
    t = re.sub('(?<!ग)्([ं])', r'\1', t)
    t = re.sub('([ा-्])़', r'\1', t)
    t = t.replace('\u0947\u093e', '\u094b').replace('\u0948\u093e', '\u094c').replace('\u094d\u0903', '\u094d')
    for a, b in (('अा', 'आ'), ('आा', 'आ'), ('अो', 'ओ'), ('अौ', 'औ'), ('अे', 'ए'), ('एे', 'ऐ'), ('अि', 'इ'), ('इि', 'इ'), ('अी', 'ई'), ('इी', 'ई'), ('उू', 'ऊ'),
                 ('\u0941\u0942', '\u0942'), ('\u0940\u093f', '\u0940'), ('\u0940\u0947', '\u0940')): t = t.replace(a, b)
    return re.sub(r'([ा-ौ])\1', r'\1', t)

In [ ]:
test_eq(fix_aksharas('श्िच'), 'श्चि'); test_eq(fix_aksharas('भक्ित'), 'भक्ति'); test_eq(fix_aksharas('ध्ार्म'), 'धार्म')
test_eq(fix_aksharas('कण्ट्ःए'), 'कण्ठे'); test_eq(fix_aksharas('अाम'), 'आम'); test_eq(fix_aksharas('ग्ं'), 'ग्ं')
test_eq(bad_aksharas('ध्ार्म'), ['virāma before a vowel sign']); test_eq(bad_aksharas(fix_aksharas('ध्ार्म')), [])

## Typing slips

Three small helpers fix slips in Devanagari typed by a person or written by a model.

`norm` turns a colon written after a Devanagari letter into a visarga (ः), writes ASCII digits between ॥ marks in Devanagari digits, and removes spaces before a daṇḍa (। or ॥). So `राम: ॥ 1 ॥` becomes `रामः॥ १॥`.

A pāda is a quarter of a verse, usually printed as a line or half line ending in a daṇḍa. A word at the end of a pāda often ends in a bare consonant marked with a virāma, as in जगत्, and typists drop the virāma. `virama` puts it back when a line ends with one of a few endings that need it, either at the end of the line or before a ।. The endings are त after ा, े, ृ, व or य (-ात्, -ेत्, -ृत्, -वत्, -यत्), न after ा or ि (-ान्, -िन्), and the whole words जगत and सत. A word before ॥ is not changed.

`deva` writes a number in Devanagari digits.

In [ ]:
#| export
_DEVA = str.maketrans('0123456789', '०१२३४५६७८९')
def deva(n) -> str:
    "Write a number in Devanagari digits: 18 becomes १८."
    return str(n).translate(_DEVA)

def norm(t:str) -> str:
    "Fix typing slips in `t`: a colon typed for a visarga, ASCII digits inside `॥ n ॥`, and spaces before a daṇḍa."
    t = re.sub(r'(?<=[ऀ-ॿ]):', 'ः', t)
    t = re.sub(r'॥\s*([0-9]+)\s*॥', lambda m: f'॥ {deva(m[1])} ॥', t)
    return re.sub(r'[ \t]+([।॥])', r'\1', t).strip()

_VIR = re.compile(r'(?<=[क-ह])(?:(?<=[ाेृवय]त)|(?<=[ाि]न)|(?<=जगत)|(?<=^सत)|(?<= सत))(?=\s*(?:।|$))')
def virama(l:str) -> str:
    "Restore the virāma a typist drops at the end of a pāda, for the endings -ात्, -ेत्, -ृत्, -वत्, -यत्, -ान्, -िन् and the words जगत्, सत्, before । or the end of the line."
    return _VIR.sub('्', l)

In [ ]:
test_eq(deva(18), '१८'); test_eq(norm('रामः वनं गच्छति:  ।\nसीता ॥ 12 ॥'), 'रामः वनं गच्छतिः।\nसीता॥ १२॥')
test_eq(virama('भगवत ।'), 'भगवत् ।'); test_eq(virama('महान'), 'महान्'); test_eq(virama('जगत'), 'जगत्'); test_eq(virama('राम ।'), 'राम ।')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()